# Multiclass Classifier Training

This notebook prepares a `50,000`-row labeled multiclass dataset and trains the wafer-defect classifier before the showcase notebook.

Workflow:
- load all labeled WM-811K multiclass rows
- keep every labeled defect wafer
- sample normal wafers until the labeled set reaches `50,000` rows total
- export arrays and metadata under `data/processed/x64/wm811k_multiclass_50k`
- optionally train the classifier and save artifacts under `experiments/classifier/multiclass/x64/training/artifacts/multiclass_classifier_50k`
- optionally score the unlabeled wafers with the trained checkpoint

Project target: push overall accuracy as high as possible, but also track balanced accuracy and class-wise recall so a high score is not coming only from easier classes.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

NOTEBOOK_DIR = REPO_ROOT / "experiments/classifier/multiclass/x64/training"

from wafer_defect.config import load_toml
from wafer_defect.classification.data import (
    DEFAULT_CLASS_NAMES,
    NORMAL_CLASS,
    build_labeled_metadata,
    prepare_supervised_dataframe,
)

In [ ]:
data_config_path = NOTEBOOK_DIR / "data_config.toml"
train_config_path = NOTEBOOK_DIR / "train_config.toml"

data_config = load_toml(data_config_path)
train_config = load_toml(train_config_path)

dataset_cfg = data_config["dataset"]
split_cfg = data_config["splits"]
training_cfg = train_config["training"]

class_names = list(dataset_cfg.get("class_names", DEFAULT_CLASS_NAMES))
raw_pickle = REPO_ROOT / dataset_cfg["raw_pickle"]
processed_root = REPO_ROOT / dataset_cfg["processed_root"]
metadata_path = REPO_ROOT / dataset_cfg["labeled_metadata_csv"]
arrays_dir = REPO_ROOT / dataset_cfg["labeled_arrays_dir"]
artifacts_dir = NOTEBOOK_DIR / "artifacts" / Path(training_cfg["output_dir"]).name
checkpoint_path = artifacts_dir / "best_model.pt"
history_path = artifacts_dir / "history.csv"
metrics_path = artifacts_dir / "metrics.json"
summary_path = processed_root / "dataset_summary.txt"
target_total_labeled = 50_000
RUN_TRAINING = False
random_seed = int(training_cfg["random_seed"])

print(f"Data config: {data_config_path}")
print(f"Train config: {train_config_path}")
print(f"Processed root: {processed_root}")
print(f"Artifacts dir: {artifacts_dir}")
print(f"Run training: {RUN_TRAINING}")

In [ ]:
if RUN_TRAINING:
    labeled_df, unlabeled_df = prepare_supervised_dataframe(raw_pickle=raw_pickle, class_names=class_names)

    defect_df = labeled_df[labeled_df["failure_type"] != NORMAL_CLASS].copy()
    normal_df = labeled_df[labeled_df["failure_type"] == NORMAL_CLASS].copy()
    normal_target = target_total_labeled - len(defect_df)

    if normal_target <= 0:
        raise ValueError(
            f"target_total_labeled={target_total_labeled} is too small because there are already {len(defect_df)} defect rows."
        )
    if normal_target > len(normal_df):
        raise ValueError(
            f"Need {normal_target} normal rows but only found {len(normal_df)} normal rows."
        )

    sampled_normal_df = normal_df.sample(n=normal_target, random_state=random_seed)
    sampled_labeled_df = pd.concat([defect_df, sampled_normal_df], ignore_index=True)
    sampled_labeled_df = sampled_labeled_df.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)

    metadata = build_labeled_metadata(
        labeled_df=sampled_labeled_df,
        class_names=class_names,
        image_size=int(dataset_cfg["image_size"]),
        arrays_dir=arrays_dir,
        metadata_path=metadata_path,
        repo_root=REPO_ROOT,
        train_fraction=float(split_cfg["train_fraction"]),
        val_fraction=float(split_cfg["val_fraction"]),
        test_fraction=float(split_cfg["test_fraction"]),
        random_seed=int(split_cfg["random_seed"]),
    )

    processed_root.mkdir(parents=True, exist_ok=True)
    summary_lines = [
        f"target_total_labeled={target_total_labeled}",
        f"labeled_rows_full={len(labeled_df)}",
        f"labeled_rows_sampled={len(sampled_labeled_df)}",
        f"unlabeled_rows={len(unlabeled_df)}",
        "",
        "class_counts:",
        metadata["label_name"].value_counts().sort_index().to_string(),
        "",
        "split_counts:",
        metadata["split"].value_counts().sort_index().to_string(),
    ]
    summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

    print(f"Saved metadata to {metadata_path}")
    print(f"Saved arrays under {arrays_dir}")
    print(f"Saved dataset summary to {summary_path}")
else:
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"RUN_TRAINING is False but the saved metadata file is missing: {metadata_path}"
        )
    metadata = pd.read_csv(metadata_path)
    print("RUN_TRAINING is False. Skipping dataset export and training.")
    print(f"Loaded existing metadata from {metadata_path}")
    if summary_path.exists():
        print(f"Using existing dataset summary: {summary_path}")

In [ ]:
summary = pd.DataFrame(
    {
        "count": metadata["label_name"].value_counts().reindex(class_names, fill_value=0),
        "train": metadata[metadata["split"] == "train"]["label_name"].value_counts().reindex(class_names, fill_value=0),
        "val": metadata[metadata["split"] == "val"]["label_name"].value_counts().reindex(class_names, fill_value=0),
        "test": metadata[metadata["split"] == "test"]["label_name"].value_counts().reindex(class_names, fill_value=0),
    }
)

display(summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
summary["count"].plot(kind="bar", ax=axes[0], color="#2F6B7A")
axes[0].set_title("50k Labeled Class Balance")
axes[0].set_ylabel("Samples")
axes[0].tick_params(axis="x", rotation=45)

metadata.groupby(["split", "label_name"]).size().unstack(fill_value=0).reindex(columns=class_names).plot(
    kind="bar", stacked=True, ax=axes[1], colormap="tab20"
)
axes[1].set_title("Split Distribution by Class")
axes[1].set_ylabel("Samples")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Class", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
if RUN_TRAINING:
    train_command = [
        sys.executable,
        str(REPO_ROOT / "scripts/classifier/train_multiclass_classifier.py"),
        "--config",
        str(train_config_path),
    ]
    print("Running:", " ".join(train_command))
    subprocess.run(train_command, cwd=REPO_ROOT, check=True)
else:
    print(f"RUN_TRAINING is False. Reusing saved training artifacts from {artifacts_dir}")

## Final Labeling

This notebook now stops after supervised training and evaluation.
Run `experiments/classifier/multiclass/x64/final_labeling/notebook.ipynb` only after you have chosen the final checkpoint you want to trust for unlabeled pseudo-labeling.

In [ ]:
required_artifact_paths = [history_path, metrics_path]
missing_artifact_paths = [path for path in required_artifact_paths if not path.exists()]
if missing_artifact_paths:
    raise FileNotFoundError(
        "Missing required training artifacts:\n" + "\n".join(str(path) for path in missing_artifact_paths)
    )

history = pd.read_csv(history_path)
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
checkpoint_metric_name = metrics.get("checkpoint_metric_name", "validation_accuracy")
checkpoint_metric_value = metrics.get("best_checkpoint_metric", metrics.get("best_val_accuracy"))

display(history.tail())
display(pd.DataFrame(metrics["test"]["classification_report"]).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
history.plot(x="epoch", y=["train_eval_accuracy", "val_accuracy"], ax=axes[0], marker="o")
axes[0].set_title("Accuracy by Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(alpha=0.3)

history.plot(x="epoch", y=["train_eval_balanced_accuracy", "val_balanced_accuracy"], ax=axes[1], marker="o")
axes[1].set_title("Balanced Accuracy by Epoch")
axes[1].set_ylabel("Balanced Accuracy")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best checkpoint metric ({checkpoint_metric_name}):", checkpoint_metric_value)
print("Test accuracy:", metrics["test"]["accuracy"])
print("Test balanced accuracy:", metrics["test"]["balanced_accuracy"])

In [ ]:
import shutil

archive_dir = NOTEBOOK_DIR / "artifacts" / "archive"
archive_dir.mkdir(parents=True, exist_ok=True)

archived_checkpoint_path = archive_dir / "14_multiclass_classifier_best_model.pt"
archived_history_path = archive_dir / "14_multiclass_classifier_history.csv"
archived_metrics_path = archive_dir / "14_multiclass_classifier_metrics.json"

for source_path, target_path in [
    (checkpoint_path, archived_checkpoint_path),
    (history_path, archived_history_path),
    (metrics_path, archived_metrics_path),
]:
    if source_path.exists():
        shutil.copy2(source_path, target_path)
        print(f"Archived {source_path.name} to {target_path}")
    else:
        print(f"Skipped missing artifact: {source_path}")

## Next Step

Open `experiments/classifier/multiclass/x64/showcase/notebook.ipynb` for presentation-oriented analysis, or `experiments/classifier/multiclass/x64/final_labeling/notebook.ipynb` when you are ready to generate unlabeled predictions from the final chosen checkpoint.